# Assignment 3: English to French machine translation
Run the cells in order on a Colab GPU runtime. All four models use the same
cleaned OPUS-100 examples and the same 1,000-sentence test set.
Results are generated by the notebook; no benchmark numbers are fabricated.


## 0. Install and configure
Restart the runtime if Colab asks after installation. A T4 supports QLoRA,
but a full 25k-pair run for all models can take longer than one Colab session.
Run one model section per session if needed; checkpoints and predictions persist
only if OUT is set to a mounted Drive directory.


In [ ]:
%pip -q install "transformers==4.56.2" "datasets==4.0.0" "evaluate==0.4.6" "peft==0.17.1" "trl==0.22.2" "accelerate==1.10.1" "bitsandbytes==0.47.0" "sacrebleu==2.5.1" "tokenizers==0.22.1" "sentencepiece==0.2.1" pandas


In [ ]:
import gc
import json
import random
import re
import time
import unicodedata
from pathlib import Path

import evaluate
import numpy as np
import pandas as pd
import torch
from datasets import Dataset, DatasetDict, load_dataset
from torch import nn
from torch.utils.data import DataLoader
from transformers import (
    AutoModelForCausalLM, AutoModelForSeq2SeqLM, AutoTokenizer,
    BitsAndBytesConfig, DataCollatorForSeq2Seq, Seq2SeqTrainer,
    Seq2SeqTrainingArguments, set_seed,
)
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model
from trl import SFTConfig, SFTTrainer
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
set_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
assert DEVICE.type == "cuda", "Select a GPU runtime for the full experiment."
# In Colab, persist outputs and checkpoints in the user's Google Drive.
# The notebook requests Drive authorization only on Colab runtimes.
try:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT = Path("/content/drive/MyDrive/FGAI_Assignment3")
except ImportError:
    OUT = Path("assignment3_outputs")
OUT.mkdir(parents=True, exist_ok=True)
TRAIN_N, VALID_N, TEST_N = 25_000, 1_000, 1_000
MAX_SRC_TOKENS, MAX_TGT_TOKENS, MAX_NEW_TOKENS = 128, 128, 96
print(torch.__version__, torch.cuda.get_device_name(), OUT.resolve())


## 1. Load, clean, and freeze a common dataset
OPUS-100 is publicly accessible, but its Hugging Face card does not identify
a single dataset-wide license. Check the underlying OPUS source licenses before
redistributing pairs or using them commercially. Cleaning removes empty,
extremely short/long, identical, and duplicate pairs. We also block repeated
English or French sentences across the three final splits.


In [ ]:
def clean_text(value):
    if not isinstance(value, str):
        return ""
    value = unicodedata.normalize("NFC", value)
    value = re.sub(r"\s+", " ", value).strip()
    return value

def candidate_pairs(raw):
    for row in raw:
        translation = row.get("translation") or {}
        en = clean_text(translation.get("en"))
        fr = clean_text(translation.get("fr"))
        if not (3 <= len(en) <= 300 and 3 <= len(fr) <= 300):
            continue
        if en.casefold() == fr.casefold():
            continue
        yield {"en": en, "fr": fr}

def select_unique(raw, count, seen_en, seen_fr):
    selected = []
    for pair in candidate_pairs(raw):
        en_key, fr_key = pair["en"].casefold(), pair["fr"].casefold()
        if en_key in seen_en or fr_key in seen_fr:
            continue
        seen_en.add(en_key)
        seen_fr.add(fr_key)
        selected.append(pair)
        if len(selected) == count:
            break
    if len(selected) != count:
        raise RuntimeError(f"Only {len(selected)} of {count} clean pairs found; increase the raw slice.")
    return Dataset.from_list(selected)

# Prefer official held-out splits. A larger randomized train slice supplies enough
# pairs after filtering; it is fixed by seed and cached for repeat runs.
raw_valid = load_dataset("Helsinki-NLP/opus-100", "en-fr", split="validation")
raw_test = load_dataset("Helsinki-NLP/opus-100", "en-fr", split="test")
raw_train = load_dataset("Helsinki-NLP/opus-100", "en-fr", split="train[:80000]").shuffle(seed=SEED)
seen_en, seen_fr = set(), set()
data = DatasetDict({
    "test": select_unique(raw_test, TEST_N, seen_en, seen_fr),
    "validation": select_unique(raw_valid, VALID_N, seen_en, seen_fr),
    "train": select_unique(raw_train, TRAIN_N, seen_en, seen_fr),
})
data.save_to_disk(str(OUT / "clean_opus100_en_fr"))
print({split: len(ds) for split, ds in data.items()})
display(pd.DataFrame(data["train"][:3]))


## 2. Shared benchmark helpers
Translation quality uses corpus BLEU and chrF. Latency includes tokenization
and generation on the same GPU, after one warm-up sentence; it excludes model
loading and metric calculation. All primary scores use greedy decoding and
at most 96 newly generated tokens. Scratch and mT5 are sequence-to-sequence;
Llama and Gemma generate only the continuation after their chat prompt.


In [ ]:
bleu_metric = evaluate.load("sacrebleu")
chrf_metric = evaluate.load("chrf")
sources = list(data["test"]["en"])
references = list(data["test"]["fr"])
results_file = OUT / "results.csv"
results = (pd.read_csv(results_file).to_dict("records")
           if results_file.exists() else [])
temperature_file = OUT / "temperature_analysis.csv"
temperature_rows = (pd.read_csv(temperature_file).to_dict("records")
                    if temperature_file.exists() else [])
TEMPERATURES = [0.0, 0.3, 0.7, 1.0]
PROBE_IDS = [0, 1, 2, 3, 4]

def sync_cuda():
    if torch.cuda.is_available():
        torch.cuda.synchronize()

def count_parameters(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable

def evaluate_and_save(name, model, translate_batch, train_minutes, batch_size):
    model.eval()
    translate_batch(sources[:1])  # GPU warm-up, excluded from timed inference.
    sync_cuda()
    start = time.perf_counter()
    predictions = []
    for offset in range(0, len(sources), batch_size):
        predictions.extend(translate_batch(sources[offset:offset + batch_size]))
    sync_cuda()
    elapsed = time.perf_counter() - start
    assert len(predictions) == len(references)
    total, trainable = count_parameters(model)
    row = {
        "model": name,
        "BLEU": bleu_metric.compute(predictions=predictions,
                                    references=[[r] for r in references])["score"],
        "chrF": chrf_metric.compute(predictions=predictions,
                                    references=references)["score"],
        "train_min": train_minutes,
        "sec_per_100": elapsed * 100 / len(sources),
        "total_params": total,
        "trainable_params": trainable,
    }
    results[:] = [r for r in results if r["model"] != name]
    results.append(row)
    pd.DataFrame({"en": sources, "reference_fr": references,
                  "prediction_fr": predictions}).to_csv(
                      OUT / f"{name}_predictions.csv", index=False)
    pd.DataFrame(results).to_csv(OUT / "results.csv", index=False)
    display(pd.DataFrame([row]).round(3))
    return predictions

def release_model(*objects):
    for obj in objects:
        del obj
    gc.collect()
    torch.cuda.empty_cache()

def record_temperature(name, translate):
    temperature_rows[:] = [r for r in temperature_rows if r["model"] != name]
    for temperature in TEMPERATURES:
        set_seed(SEED)  # Fixed seed for reproducible sampled comparisons.
        for idx in PROBE_IDS:
            translation = translate([sources[idx]], temperature=temperature)[0]
            temperature_rows.append({
                "model": name, "temperature": temperature,
                "english": sources[idx], "reference_french": references[idx],
                "generated_french": translation,
            })
    pd.DataFrame(temperature_rows).to_csv(temperature_file, index=False)


## 3. Model 1: custom BPE + Transformer encoder-decoder
The BPE is trained only on the training split. The model uses teacher forcing,
learned position embeddings, padding masks, and a causal decoder mask.


In [ ]:
SPECIAL = ["[PAD]", "[UNK]", "[BOS]", "[EOS]"]
bpe = Tokenizer(models.BPE(unk_token="[UNK]"))
bpe.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
bpe.decoder = decoders.ByteLevel()
bpe.train_from_iterator(
    (text for pair in data["train"] for text in (pair["en"], pair["fr"])),
    trainer=trainers.BpeTrainer(vocab_size=16_000, special_tokens=SPECIAL,
                               initial_alphabet=pre_tokenizers.ByteLevel.alphabet()),
    length=2 * TRAIN_N,
)
bpe.save(str(OUT / "scratch_bpe.json"))
PAD, UNK, BOS, EOS = (bpe.token_to_id(x) for x in SPECIAL)
VOCAB = bpe.get_vocab_size()

def encode_bpe(text, max_len):
    return [BOS] + bpe.encode(text).ids[:max_len - 2] + [EOS]

def scratch_collate(batch):
    src = [torch.tensor(encode_bpe(r["en"], MAX_SRC_TOKENS)) for r in batch]
    tgt = [torch.tensor(encode_bpe(r["fr"], MAX_TGT_TOKENS)) for r in batch]
    return (nn.utils.rnn.pad_sequence(src, batch_first=True, padding_value=PAD),
            nn.utils.rnn.pad_sequence(tgt, batch_first=True, padding_value=PAD))

train_loader = DataLoader(data["train"], batch_size=32, shuffle=True,
                          collate_fn=scratch_collate, pin_memory=True)
valid_loader = DataLoader(data["validation"], batch_size=32, shuffle=False,
                          collate_fn=scratch_collate, pin_memory=True)

class ScratchTransformer(nn.Module):
    def __init__(self, vocab, d_model=256, nhead=4, layers=3, ff=1024, max_pos=512):
        super().__init__()
        self.d_model = d_model
        self.embedding = nn.Embedding(vocab, d_model, padding_idx=PAD)
        self.position = nn.Embedding(max_pos, d_model)
        self.net = nn.Transformer(d_model=d_model, nhead=nhead,
                                  num_encoder_layers=layers, num_decoder_layers=layers,
                                  dim_feedforward=ff, dropout=0.1, batch_first=True)
        self.output = nn.Linear(d_model, vocab)

    def embed(self, ids):
        positions = torch.arange(ids.size(1), device=ids.device)[None, :]
        return self.embedding(ids) * (self.d_model ** 0.5) + self.position(positions)

    def forward(self, src, tgt_in):
        tgt_mask = nn.Transformer.generate_square_subsequent_mask(
            tgt_in.size(1), device=tgt_in.device)
        hidden = self.net(self.embed(src), self.embed(tgt_in),
                          tgt_mask=tgt_mask, src_key_padding_mask=src.eq(PAD),
                          tgt_key_padding_mask=tgt_in.eq(PAD),
                          memory_key_padding_mask=src.eq(PAD))
        return self.output(hidden)

scratch = ScratchTransformer(VOCAB).to(DEVICE)
optimizer = torch.optim.AdamW(scratch.parameters(), lr=3e-4, weight_decay=0.01)
loss_fn = nn.CrossEntropyLoss(ignore_index=PAD)

def scratch_epoch(loader, training):
    scratch.train(training)
    loss_sum, tokens = 0.0, 0
    for src, tgt in loader:
        src, tgt = src.to(DEVICE), tgt.to(DEVICE)
        if training:
            optimizer.zero_grad(set_to_none=True)
        with torch.set_grad_enabled(training):
            logits = scratch(src, tgt[:, :-1])
            target = tgt[:, 1:]
            loss = loss_fn(logits.reshape(-1, VOCAB), target.reshape(-1))
            if training:
                loss.backward()
                nn.utils.clip_grad_norm_(scratch.parameters(), 1.0)
                optimizer.step()
        count = target.ne(PAD).sum().item()
        loss_sum += loss.item() * count
        tokens += count
    return loss_sum / tokens

start = time.perf_counter()
best_val = float("inf")
for epoch in range(1, 6):
    train_loss = scratch_epoch(train_loader, True)
    val_loss = scratch_epoch(valid_loader, False)
    print(f"epoch {epoch}: train NLL={train_loss:.3f}; validation NLL={val_loss:.3f}")
    if val_loss < best_val:
        best_val = val_loss
        torch.save(scratch.state_dict(), OUT / "scratch_best.pt")
scratch_train_min = (time.perf_counter() - start) / 60
scratch.load_state_dict(torch.load(OUT / "scratch_best.pt", map_location=DEVICE,
                                   weights_only=True))

@torch.inference_mode()
def scratch_translate(texts, temperature=0.0):
    src = nn.utils.rnn.pad_sequence(
        [torch.tensor(encode_bpe(s, MAX_SRC_TOKENS)) for s in texts],
        batch_first=True, padding_value=PAD).to(DEVICE)
    tgt = torch.full((len(texts), 1), BOS, device=DEVICE, dtype=torch.long)
    finished = torch.zeros(len(texts), dtype=torch.bool, device=DEVICE)
    for _ in range(MAX_NEW_TOKENS):
        logits = scratch(src, tgt)[:, -1, :]
        logits[:, PAD] = -float("inf")
        logits[:, BOS] = -float("inf")
        if temperature == 0:
            next_token = logits.argmax(-1)
        else:
            next_token = torch.multinomial(torch.softmax(logits / temperature, -1), 1).squeeze(1)
        next_token = torch.where(finished, EOS, next_token)
        tgt = torch.cat([tgt, next_token[:, None]], dim=1)
        finished |= next_token.eq(EOS)
        if finished.all():
            break
    outputs = []
    for ids in tgt[:, 1:].tolist():
        ids = ids[:ids.index(EOS)] if EOS in ids else ids
        outputs.append(bpe.decode(ids).strip())
    return outputs

scratch_predictions = evaluate_and_save(
    "scratch", scratch, lambda xs: scratch_translate(xs, 0.0), scratch_train_min, 16)
record_temperature("scratch", scratch_translate)
del scratch, optimizer
gc.collect()
torch.cuda.empty_cache()


## 4. Model 2: mT5-small full fine-tuning
Prefixing the source makes the direction explicit. Dynamic padding avoids
wasting GPU memory. Validation loss chooses the best checkpoint.


In [ ]:
MT5_ID = "google/mt5-small"
mt5_tokenizer = AutoTokenizer.from_pretrained(MT5_ID)
mt5_model = AutoModelForSeq2SeqLM.from_pretrained(MT5_ID).to(DEVICE)

def tokenize_mt5(batch):
    source = ["translate English to French: " + s for s in batch["en"]]
    inputs = mt5_tokenizer(source, max_length=MAX_SRC_TOKENS, truncation=True)
    labels = mt5_tokenizer(text_target=batch["fr"],
                           max_length=MAX_TGT_TOKENS, truncation=True)
    inputs["labels"] = labels["input_ids"]
    return inputs

mt5_data = data.map(tokenize_mt5, batched=True,
                    remove_columns=["en", "fr"])
mt5_args = Seq2SeqTrainingArguments(
    output_dir=str(OUT / "mt5_checkpoints"),
    num_train_epochs=1, per_device_train_batch_size=4,
    per_device_eval_batch_size=8, gradient_accumulation_steps=4,
    learning_rate=3e-4, weight_decay=0.01,
    eval_strategy="epoch", save_strategy="epoch",
    load_best_model_at_end=True, metric_for_best_model="eval_loss",
    greater_is_better=False, save_total_limit=2,
    fp16=True, report_to="none", logging_steps=100,
)
mt5_trainer = Seq2SeqTrainer(
    model=mt5_model, args=mt5_args,
    train_dataset=mt5_data["train"], eval_dataset=mt5_data["validation"],
    data_collator=DataCollatorForSeq2Seq(mt5_tokenizer, model=mt5_model),
    processing_class=mt5_tokenizer,
)
start = time.perf_counter()
mt5_trainer.train()
mt5_train_min = (time.perf_counter() - start) / 60
mt5_trainer.save_model(str(OUT / "mt5_final"))
mt5_tokenizer.save_pretrained(str(OUT / "mt5_final"))
mt5_model = mt5_trainer.model

@torch.inference_mode()
def mt5_translate(texts, temperature=0.0):
    inputs = mt5_tokenizer(["translate English to French: " + s for s in texts],
                           return_tensors="pt", padding=True, truncation=True,
                           max_length=MAX_SRC_TOKENS).to(DEVICE)
    options = {"max_new_tokens": MAX_NEW_TOKENS, "num_beams": 1,
               "do_sample": temperature > 0}
    if temperature > 0:
        options.update(temperature=temperature, top_k=0, top_p=1.0)
    ids = mt5_model.generate(**inputs, **options)
    return [s.strip() for s in mt5_tokenizer.batch_decode(ids, skip_special_tokens=True)]

mt5_predictions = evaluate_and_save(
    "mt5", mt5_model, lambda xs: mt5_translate(xs, 0.0), mt5_train_min, 16)
record_temperature("mt5", mt5_translate)
del mt5_model, mt5_trainer
gc.collect()
torch.cuda.empty_cache()


## 5. Models 3 and 4: Llama and Gemma with 4-bit QLoRA
Accept each model's license in your Hugging Face account first. In Colab,
store HF_TOKEN in Secrets, or run `huggingface-cli login` interactively.
Prompt-completion datasets let TRL mask the prompt, so loss is calculated on
the French answer only. The model's own chat template is used at inference.


In [ ]:
import os
from huggingface_hub import login, notebook_login

try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
except Exception:
    hf_token = os.environ.get("HF_TOKEN")
if hf_token:
    login(token=hf_token, add_to_git_credential=False)
else:
    notebook_login()  # Interactive fallback; use a token with gated-model access.

TRANSLATION_INSTRUCTION = (
    "Translate the following English sentence into French. "
    "Return only the French translation, preserving names, numbers, and tone.\n\nEnglish: {source}"
)

def make_chat_dataset(tokenizer, split):
    rows = []
    for pair in data[split]:
        user = [{"role": "user", "content": TRANSLATION_INSTRUCTION.format(source=pair["en"])}]
        prompt = tokenizer.apply_chat_template(user, tokenize=False,
                                               add_generation_prompt=True)
        full = tokenizer.apply_chat_template(
            user + [{"role": "assistant", "content": pair["fr"]}],
            tokenize=False, add_generation_prompt=False)
        if not full.startswith(prompt):
            raise ValueError("Chat template prompt is not a prefix of the full conversation")
        rows.append({"prompt": prompt, "completion": full[len(prompt):]})
    return Dataset.from_list(rows)

def train_and_evaluate_causal(model_id, short_name):
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    tokenizer.padding_side = "right"  # training padding
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                              bnb_4bit_use_double_quant=True,
                              bnb_4bit_compute_dtype=torch.float16)
    model = AutoModelForCausalLM.from_pretrained(
        model_id, quantization_config=quant, device_map={"": 0},
        torch_dtype=torch.float16, attn_implementation="eager")
    model.config.use_cache = False
    model = prepare_model_for_kbit_training(model)
    model = get_peft_model(model, LoraConfig(
        r=16, lora_alpha=32, lora_dropout=0.05, bias="none",
        task_type="CAUSAL_LM", target_modules=["q_proj", "k_proj", "v_proj", "o_proj"]))
    train_set = make_chat_dataset(tokenizer, "train")
    valid_set = make_chat_dataset(tokenizer, "validation")
    args = SFTConfig(
        output_dir=str(OUT / f"{short_name}_checkpoints"),
        max_length=256, completion_only_loss=True, packing=False,
        num_train_epochs=1, per_device_train_batch_size=2,
        per_device_eval_batch_size=2, gradient_accumulation_steps=8,
        learning_rate=2e-4, lr_scheduler_type="cosine", warmup_ratio=0.03,
        optim="paged_adamw_8bit", fp16=True, gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        eval_strategy="epoch", save_strategy="epoch", save_total_limit=2,
        report_to="none", logging_steps=100,
    )
    trainer = SFTTrainer(model=model, args=args, train_dataset=train_set,
                         eval_dataset=valid_set, processing_class=tokenizer)
    start = time.perf_counter()
    trainer.train()
    train_minutes = (time.perf_counter() - start) / 60
    trainer.save_model(str(OUT / f"{short_name}_adapter"))
    tokenizer.save_pretrained(str(OUT / f"{short_name}_adapter"))
    model = trainer.model
    model.config.use_cache = True
    model.eval()
    tokenizer.padding_side = "left"  # batched autoregressive generation

    @torch.inference_mode()
    def translate(texts, temperature=0.0):
        prompts = [tokenizer.apply_chat_template(
            [{"role": "user", "content": TRANSLATION_INSTRUCTION.format(source=s)}],
            tokenize=False, add_generation_prompt=True) for s in texts]
        encoded = tokenizer(prompts, return_tensors="pt", padding=True,
                            truncation=True, max_length=160,
                            add_special_tokens=False).to(DEVICE)
        # Stop at the model's end-of-turn token as well as its normal EOS.
        turn_token = "<|eot_id|>" if "llama" in short_name else "<end_of_turn>"
        turn_id = tokenizer.convert_tokens_to_ids(turn_token)
        eos_ids = [tokenizer.eos_token_id]
        if isinstance(turn_id, int) and turn_id != tokenizer.unk_token_id:
            eos_ids.append(turn_id)
        options = {"max_new_tokens": MAX_NEW_TOKENS, "num_beams": 1,
                   "do_sample": temperature > 0,
                   "pad_token_id": tokenizer.pad_token_id,
                   "eos_token_id": eos_ids}
        if temperature > 0:
            options.update(temperature=temperature, top_k=0, top_p=1.0)
        generated = model.generate(**encoded, **options)
        continuation = generated[:, encoded["input_ids"].shape[1]:]
        return [x.strip() for x in tokenizer.batch_decode(
            continuation, skip_special_tokens=True)]

    predictions = evaluate_and_save(
        short_name, model, lambda xs: translate(xs, 0.0), train_minutes, 4)
    return trainer, model, tokenizer, translate, predictions


In [ ]:
llama_trainer, llama_model, llama_tokenizer, llama_translate, llama_predictions = (
    train_and_evaluate_causal("TinyLlama/TinyLlama-1.1B-Chat-v1.0", "llama"))


Run the temperature analysis for Llama now, then free its GPU memory before
loading Gemma. With OUT on Drive, the four model sections can be run in
separate Colab sessions after rerunning sections 0-2.


In [ ]:
record_temperature("llama", llama_translate)
display(pd.DataFrame(temperature_rows).tail(20))


In [ ]:
del llama_trainer, llama_model, llama_tokenizer, llama_translate
gc.collect()
torch.cuda.empty_cache()


In [ ]:
gemma_trainer, gemma_model, gemma_tokenizer, gemma_translate, gemma_predictions = (
    train_and_evaluate_causal("google/gemma-2-2b-it", "gemma"))
record_temperature("gemma", gemma_translate)


## 6. Final comparison and temperature analysis
Temperatures above zero use sampling with top_k=0 and top_p=1.0; T=0 is
deterministic greedy decoding. Sampling can yield a different valid French
sentence, so BLEU/chrF are reported only for the deterministic benchmark.
Inspect pronouns, names, numbers, omissions, and tu/vous choices manually.


In [ ]:
comparison = pd.DataFrame(results).set_index("model")
display(comparison.round({"BLEU": 2, "chrF": 2, "train_min": 2,
                          "sec_per_100": 2}))
display(pd.DataFrame(temperature_rows))
print("Saved results, predictions, and temperature outputs to", OUT.resolve())


## 7. Responsible AI and ethical analysis
**Gender and social bias.** English occupation nouns and singular “they” can
leave gender unspecified. French often demands a grammatical choice, and a
model may default to a stereotype (for example, doctor/il or nurse/elle).
Audit balanced profession and pronoun templates with human review; preserve
ambiguity where French permits it, or request context rather than inventing
identity. Report disaggregated error rates and avoid treating BLEU as a bias
measure.

**Hallucinations and omissions.** Decoder-only models may add explanations,
invent entities, or silently drop negation, numbers, and named people. Use a
constrained instruction, a fixed output limit, and post-generation checks for
numbers, names, negation, and language. Flag failures for human correction;
do not silently publish a high-stakes translation.

**Cultural and register errors.** “You” may map to informal “tu” or formal
“vous”, which conveys social distance and can change a message's meaning.
Supply audience and register metadata in production prompts, evaluate both
formal and informal test cases, and ask for clarification when the context is
missing. Preserve culturally specific names and idioms; review localized
renderings with speakers of the target variety of French.

**Sensitive content and privacy.** Parallel corpora may contain personal
details, slurs, medical statements, or threats. Translation should preserve
meaning without endorsing harmful content. Restrict data access, inspect
licenses and provenance, minimize retained personal data, and use trained
human reviewers for sensitive outputs. Never use this notebook's output as
unreviewed medical, legal, or emergency advice.

**Limits of this experiment.** OPUS-100 mixes domains and its Hugging Face
card leaves corpus-level licensing unspecified. The 1,000 examples estimate
quality only on this filtered subset. BLEU and chrF reward reference overlap,
not factual fidelity, fairness, politeness, or safety. The pretrained models
also bring different prior knowledge, so this is a practical system comparison
rather than an architecture-only controlled experiment.
